<a href="https://colab.research.google.com/github/svyatoslavna/ml_hw/blob/main/cnn__hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: обучение CNN на собранном корпусе данных

## Задача

1. Собрать свой текстовый корпус (не менее 500 примеров) с помощью библиотеки `requests` или другого инструмента парсинга.
2. Выполнить разметку собранных данных для задачи классификации (бинарной или многоклассовой).
3. Обучить свёрточную нейросеть (CNN) на собранных и размеченных данных с использованием PyTorch или TensorFlow на выбор.
4. Самостоятельно подобрать архитектуру модели и гиперпараметры (количество слоёв, размер ядра, функцию активации, оптимизатор и т.д.).
5. После обучения вывести на экран метрики качества (accuracy, precision, recall, F1-score) на тестовой выборке.

## Критерии оценки (10-балльная шкала)

| Баллы | Критерий |
|-------|----------|
| 1 | Корпус собран, объём ≥ 500 примеров |
| 2 | Корпус размечен (указаны классы для каждого примера) |
| 3 | Данные предобработаны и разделены на обучающую/валидационную/тестовую выборки |
| 4 | Модель CNN реализована (TensorFlow или PyTorch) и запущена на обучение |
| 5 | Обучение завершено без ошибок, модель сохраняет веса |
| 6 | Выведены базовые метрики (accuracy) на тестовой выборке |
| 7 | Выведен полный набор метрик: precision, recall, F1-score |
| 8 | Гиперпараметры подобраны обоснованно (с комментариями, почему выбраны именно такие значения) |
| 9 | Проведён анализ результатов (что получилось хорошо, что можно улучшить) |
| 10 | Код полностью воспроизводим, содержит комментарии, структура соответствует лучшим практикам |

## Важно

- Качество модели (точность) не является основным критерием — важнее корректность выполнения всех этапов.
- Выберите **свой уникальный источник данных** для парсинга и укажите его в отчёте.
- Разметка должна быть выполнена вручную или с помощью автоматических правил (например, по ключевым словам или меткам с сайта). Укажите способ разметки.
- Не используйте готовые датасеты из интернета (например, IMDB, MNIST, CIFAR).
- Для текстовых данных CNN обычно работает на уровне слов или символов — обоснуйте выбранный подход.

## Рекомендации по выполнению

1. **Выбор источника данных:** новостной сайт с категориями, отзывы пользователей (позитивные/негативные), посты с форума по разным темам, заголовки новостей с метками и т.д.
2. **Разметка:** можно использовать структуру сайта (например, раздел "спорт" — класс 0, раздел "политика" — класс 1) или выполнить разметку вручную для небольшого корпуса.
3. **Архитектура CNN для текста:** Embedding → Conv1D → GlobalMaxPooling1D → Dense → Dropout → Dense(классы).
4. **Гиперпараметры для подбора:** размер эмбеддингов (50–300), количество фильтров (32–256), размер ядра (3–5), dropout rate (0.3–0.7), скорость обучения.
5. **Минимальный размер корпуса:** 500 размеченных примеров. При недостаточном количестве данных используйте аугментацию или меньшую модель.

## Установка и импорт библиотек

In [85]:
!pip install beautifulsoup4 requests lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, TensorDataset

## Парсинг текстового корпуса

**Cайт:** [https://www.avito.ru/](https://www.avito.ru/)

**Обоснование выбора:** для классификации объявлений - одежда, мебель и т.д.

In [ ]:
def scrape_corpus(base_url, num_pages=5):
    """
    Парсинг текстового корпуса с выбранного сайта.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    texts = []

    for page in range(1, num_pages + 1):
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}?p={page}"

        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # находим все блоки объявлений
            items = soup.find_all('div', attrs={'data-marker': 'item'})

            print(f"Страница {page}: найдено {len(items)} объявлений")

            for item in items:
                # заголовок
                title_elem = item.find('a', attrs={'data-marker': 'item-title'})
                title = title_elem.text.strip() if title_elem else ''

                # цена
                price_meta = item.find('meta', attrs={'itemprop': 'price'})
                price = price_meta.get('content') if price_meta else ''

                # описание (но оно на страничке не полностью отражается:((((()
                desc_meta = item.find('meta', attrs={'itemprop': 'description'})
                description = desc_meta.get('content') if desc_meta else ''

                # сохраняем только если есть заголовок
                if title:
                    texts.append({
                        'title': title,
                        'price': price,
                        'description': description
                    })

            print(f"  Всего собрано за всё время: {len(texts)} записей")

            time.sleep(3)

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")

    return texts

In [ ]:
# список страничек с разными категориями товаров

categories = {'Игрушки и товары для детей': 'https://www.avito.ru/sankt-peterburg/tovary_dlya_detey_i_igrushki',
            'Готовый бизнес и оборудование для бизнеса': 'https://www.avito.ru/sankt-peterburg/dlya_biznesa',
            'Женская одежда': 'https://www.avito.ru/sankt-peterburg/odezhda_obuv_aksessuary/zhenskaya_odezhda-ASgBAgICAUTeAtYL',
            'Аксессуары': 'https://www.avito.ru/sankt-peterburg/odezhda_obuv_aksessuary/kupit-aksessuary-ASgBAgICAUTeAtoL',
            'Сумки, рюкзаки и чемоданы': 'https://www.avito.ru/sankt-peterburg/odezhda_obuv_aksessuary/sumki_ryukzaki_i_chemodany-ASgBAgICAUTeArip1gI',
            'Хобби и отдых': 'https://www.avito.ru/sankt-peterburg/hobbi_i_otdyh',
            'Красота и здоровье': 'https://www.avito.ru/sankt-peterburg/krasota_i_zdorove'}

corpus = {} # сюда будем собирать все данные

for cat_name, cat_url in categories.items():
    corpus[cat_name] = scrape_corpus(cat_url, num_pages=3)
    print(f"Всего собрано в категории '{cat_name}': {len(corpus[cat_name])}")
    time.sleep(5)

Страница 1: найдено 81 объявлений
  Всего собрано за всё время: 81 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё время: 131 записей
Страница 3: найдено 50 объявлений
  Всего собрано за всё время: 181 записей
Всего собрано в категории 'Игрушки и товары для детей': 181
Страница 1: найдено 50 объявлений
  Всего собрано за всё время: 50 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё время: 100 записей
Страница 3: найдено 50 объявлений
  Всего собрано за всё время: 150 записей
Всего собрано в категории 'Готовый бизнес и оборудование для бизнеса': 150
Страница 1: найдено 81 объявлений
  Всего собрано за всё время: 81 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё время: 131 записей
Страница 3: найдено 50 объявлений
  Всего собрано за всё время: 181 записей
Всего собрано в категории 'Женская одежда': 181
Страница 1: найдено 81 объявлений
  Всего собрано за всё время: 81 записей
Страница 2: найдено 50 объявлений
  Всего собрано за всё врем

In [ ]:
total = 0
for cat_name, items in corpus.items():
    total += len(items)

print(f"Всего объявлений по всем категориям: {total}")

Всего объявлений по всем категориям: 1169


In [ ]:
# оформляем датасет

dataset = []

for cat_name, items in corpus.items():
    for item in items:
        dataset.append({
            'category': cat_name,
            'title': item['title'],
            'price': item['price'],
            'description': item['description']
        })

df = pd.DataFrame(dataset)

print(f"Датасет создан: {len(df)} строк")
print(f"Колонки: {df.columns.tolist()}")

Датасет создан: 1169 строк
Колонки: ['category', 'title', 'price', 'description']


In [ ]:
df.to_csv('avito_dataset.csv', index=False, encoding='utf-8') # сохраняем файл

# Подготовка данных к обучению

In [100]:
df = pd.read_csv('avito_dataset.csv') # выгружаем датасет
df

,category,title,price,description
0,Игрушки и товары для детей,Шкаф в детскую,10099,Шкаф в детскую на заказ.\n\nБесплатно:\n\nДост...
1,Игрушки и товары для детей,"Пододеяльник 1,5 сп.Ромб. Бязь ГОСТ 100%хлопок",455,Пододеяльник с ромбом 145х210.
2,Игрушки и товары для детей,Шкаф в детскую на заказ,9900,Мебель для дома по индивидуальным проектам под...
3,Игрушки и товары для детей,Стильный шкаф в детскую,9090,Шкаф на заказ.\n\nБесплатно.\n\nДоставляем!\n\...
4,Игрушки и товары для детей,Шкаф на заказ в детскую,9900,Мебель для дома по индивидуальным проектам под...
...,...,...,...,...
1164,Красота и здоровье,Аренда коляски инвалидной с электроприводом,1000,Если вам нужна максимальная свобода в передвиж...
1165,Красота и здоровье,Liquides Imaginaires Blanche Bete распив,2600,Аромат из личной коллекции.\n\nLes Liquides Im...
1166,Красота и здоровье,Бахилы одноразовые оптом,79,Цены ниже рынка! Скидки для ИП и Юрлиц. 2000+ ...
1167,Красота и здоровье,Легендарные кремы французской аптеки,2650,"Faq:\nАкне у подростков, пигментация, омоложен..."


In [101]:
df = df[['description', 'category']].dropna()
df.head()

,description,category
0,Шкаф в детскую на заказ.\n\nБесплатно:\n\nДост...,Игрушки и товары для детей
1,Пододеяльник с ромбом 145х210.,Игрушки и товары для детей
2,Мебель для дома по индивидуальным проектам под...,Игрушки и товары для детей
3,Шкаф на заказ.\n\nБесплатно.\n\nДоставляем!\n\...,Игрушки и товары для детей
4,Мебель для дома по индивидуальным проектам под...,Игрушки и товары для детей


In [102]:
df['category'].value_counts() # всего

,count
category,
Игрушки и товары для детей,181
Женская одежда,181
Аксессуары,181
"Сумки, рюкзаки и чемоданы",176
Готовый бизнес и оборудование для бизнеса,150
Хобби и отдых,150
Красота и здоровье,150


In [104]:
df['category'] = df['category'].astype("category").cat.codes
df

,description,category
0,Шкаф в детскую на заказ.\n\nБесплатно:\n\nДост...,3
1,Пододеяльник с ромбом 145х210.,3
2,Мебель для дома по индивидуальным проектам под...,3
3,Шкаф на заказ.\n\nБесплатно.\n\nДоставляем!\n\...,3
4,Мебель для дома по индивидуальным проектам под...,3
...,...,...
1164,Если вам нужна максимальная свобода в передвиж...,4
1165,Аромат из личной коллекции.\n\nLes Liquides Im...,4
1166,Цены ниже рынка! Скидки для ИП и Юрлиц. 2000+ ...,4
1167,"Faq:\nАкне у подростков, пигментация, омоложен...",4


In [72]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['description']).toarray()
y = df['category'].values

In [73]:
X.shape

(1169, 9587)

In [74]:
X[0][1500:1600]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [75]:
y

array([3, 3, 3, ..., 4, 4, 4], dtype=int8)

In [76]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}\nTest: {X_test.shape}")

Train: (935, 9587)
Test: (234, 9587)


In [77]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.int64)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.int64)

X_train_tensor[0], y_train_tensor[0]

(tensor([0., 0., 0.,  ..., 0., 0., 0.]), tensor(2))

In [78]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)
X_test_tensor = X_test_tensor.to(device)
y_test_tensor = y_test_tensor.to(device)

In [79]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [80]:
train_dataset # контейнер данных

In [81]:
train_loader # итератор: он оборачивает Dataset и добавляет batching, shuffling, parallel loading

# Обучение

## Модель 1


Я попробовала следовать архитектуре из задания:
> Embedding → Conv1D → GlobalMaxPooling1D → Dense → Dropout → Dense(классы).

Вместо слоя Embedding у меня CountVectorizer().

Но получалось не очень хорошо, что видно в выводе следующей ячейки.

In [112]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)

        self.global_pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = CNN1D(X.shape[1], len(np.unique(y))).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.9503
Epoch 2, Loss: 1.9449
Epoch 3, Loss: 1.9391
Epoch 4, Loss: 1.9439
Epoch 5, Loss: 1.9355
Epoch 6, Loss: 1.9346
Epoch 7, Loss: 1.9286
Epoch 8, Loss: 1.9188
Epoch 9, Loss: 1.9254
Epoch 10, Loss: 1.9234


In [113]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    _, predicted = torch.max(predictions, 1)

In [114]:
print(classification_report(y_test, predicted.cpu().numpy()))
print(confusion_matrix(y_test, predicted.cpu().numpy()))

              precision    recall  f1-score   support

           0       0.19      0.43      0.27        37
           1       0.39      0.40      0.39        30
           2       0.00      0.00      0.00        36
           3       0.14      0.25      0.18        36
           4       0.00      0.00      0.00        30
           5       0.00      0.00      0.00        35
           6       0.24      0.43      0.31        30

    accuracy                           0.21       234
   macro avg       0.14      0.22      0.16       234
weighted avg       0.13      0.21      0.16       234

[[16  3  0 12  0  0  6]
 [ 4 12  0  7  0  0  7]
 [15  6  0  4  0  0 11]
 [17  2  0  9  0  0  8]
 [11  5  0 10  0  0  4]
 [13  1  0 16  0  0  5]
 [ 7  2  0  8  0  0 13]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Модель 2

Поэтому я попробовала использовать более простую архитектуру из туториала, но несколько улучшила и её.

In [87]:
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(128 * (input_dim // 8), 256)
        self.fc2 = nn.Linear(256, num_classes)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

model = CNN1D(X.shape[1], len(np.unique(y))).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.8364
Epoch 2, Loss: 0.5557
Epoch 3, Loss: 0.1170
Epoch 4, Loss: 0.0429
Epoch 5, Loss: 0.0227
Epoch 6, Loss: 0.0176
Epoch 7, Loss: 0.0069
Epoch 8, Loss: 0.0059
Epoch 9, Loss: 0.0063
Epoch 10, Loss: 0.0059


## 7. Получаем предсказания

In [109]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    _, predicted = torch.max(predictions, 1)

In [111]:
print(classification_report(y_test, predicted.cpu().numpy()))
print(confusion_matrix(y_test, predicted.cpu().numpy()))

              precision    recall  f1-score   support

           0       0.61      0.59      0.60        37
           1       0.88      0.70      0.78        30
           2       0.73      0.89      0.80        36
           3       0.82      0.75      0.78        36
           4       0.79      0.63      0.70        30
           5       0.81      0.97      0.88        35
           6       0.55      0.57      0.56        30

    accuracy                           0.74       234
   macro avg       0.74      0.73      0.73       234
weighted avg       0.74      0.74      0.73       234

[[22  1  3  2  1  4  4]
 [ 2 21  1  0  2  1  3]
 [ 1  0 32  0  0  1  2]
 [ 2  2  0 27  0  1  4]
 [ 6  0  3  1 19  0  1]
 [ 1  0  0  0  0 34  0]
 [ 2  0  5  3  2  1 17]]


В итоге я взяла более простую архитектуру из туториала и увеличила количество свёрточных слоёв (до 3) - чтобы модель находила чуть более глубокие паттерны, количество фильтров (32 > 64 > 128) - чтобы модель находила больше важных паттернов, размер полносвязного слоя (128, 256) - чтобы она смогла обработать все найденные признаки, и количество эпох (до 10) - чтобы у модели было больше времени на обучение.

----

В целом, результат нормальный (Accuracy=74%), но мог быть и лучше, особенно на определённых классах. Что можно было бы улучшить:

- сбалансировать каким-то образом классы, возможно, из-за дисбаланса классы сильно различаются по метрикам

- добавить validation выборку для более удобного контроля обучения модели

- доработать архитектуру, включить DropOut, GlobalMaxPooling - но с другими значениями, и поперебирать все остальные гиперпараметры
---


---


Использование ИИ:

ИИ помог разобраться с парсингом, подсказал, какие метки искать в html-коде. Помог с этой частью кода:
```python
for item in items:
    # заголовок
    title_elem = item.find('a', attrs={'data-marker': 'item-title'})
    title = title_elem.text.strip() if title_elem else ''

    # цена
    price_meta = item.find('meta', attrs={'itemprop': 'price'})
    price = price_meta.get('content') if price_meta else ''

    # описание (но оно на страничке не полностью отражается:((()
    desc_meta = item.find('meta', attrs={'itemprop': 'description'})
    description = desc_meta.get('content') if desc_meta else ''
```

Также ИИ помог найти GlobalMaxPooling в PyTorch, т.к. там он под другим названием.